In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers = {"User-Agent": "Mozilla/5.0 (pesquisa academica - IC UFCAT)"}

url_listagem = "https://www.boatos.org/"
resposta = requests.get(url_listagem, headers=headers)
print("Status:", resposta.status_code)

soup = BeautifulSoup(resposta.text, "html.parser")
links = [a["href"] for a in soup.select("h2.blog-entry-title a")]
print(len(links), "links encontrados")
print(links[:3])

Status: 200
25 links encontrados
['https://www.boatos.org/brasil/coca-cola-fechou-fabricas-no-brasil-por-causa-de-lula-e-do-pt.html', 'https://www.boatos.org/esporte/zico-anuncia-voto-em-lula-no-segundo-turno-das-eleicoes-de-2026.html', 'https://www.boatos.org/entretenimento/virginia-fonseca-anuncia-voto-em-lula-nas-eleicoes-de-2026.html']


In [ ]:
url_noticia = links[0]
resposta = requests.get(url_noticia, headers=headers)
soup = BeautifulSoup(resposta.text, "html.parser")

# Procura o elemento que contém mais parágrafos (provavelmente o corpo da notícia)
melhor = None
maior = 0
for el in soup.find_all(["div", "article", "section"]):
    qtd = len(el.find_all("p", recursive=False))
    if qtd > maior:
        maior = qtd
        melhor = el

print("Título:", soup.select_one("h1").get_text(strip=True))
print("Elemento do texto:", melhor.name, melhor.get("class"))
print("Parágrafos diretos:", maior)
print(melhor.get_text(" ", strip=True)[:500])

Título: Notícia sobre fábricas da Coca-Cola fechadas não tem relação com unidades da empresa no Brasil
Elemento do texto: div ['nv-content-wrap', 'entry-content']
Parágrafos diretos: 15
Boato – A Coca-Cola fechou quatro fábricas no Brasil e demitiu cerca de 1.400 funcionários por causa do governo Lula e do PT. Análise Publicações nas redes sociais passaram a compartilhar notícias sobre o fechamento de fábricas da Coca-Cola e a demissão de funcionários, acompanhadas de comentários críticos ao governo do presidente Luiz Inácio Lula da Silva. Algumas mensagens sugerem que os encerramentos teriam ocorrido no Brasil e seriam consequência da situação econômica do país. Uma das versõe


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0 (pesquisa academica - IC UFCAT)"}

NUM_PAGINAS = 2  # comece com 2 pra testar, depois aumente

def pegar_links(url_listagem):
    resposta = requests.get(url_listagem, headers=headers)
    if resposta.status_code != 200:
        return []
    soup = BeautifulSoup(resposta.text, "html.parser")
    return [a["href"] for a in soup.select("h2.blog-entry-title a")]

def pegar_noticia(url):
    resposta = requests.get(url, headers=headers)
    soup = BeautifulSoup(resposta.text, "html.parser")

    titulo = soup.select_one("h1").get_text(strip=True)

    corpo = soup.select_one("div.entry-content")
    paragrafos = [p.get_text(" ", strip=True) for p in corpo.find_all("p")]
    texto_completo = " ".join(paragrafos)

    # o primeiro paragrafo que comeca com "Boato" e o boato original em si
    boato = next((p for p in paragrafos if p.startswith("Boato")), paragrafos[0])

    tag_data = soup.select_one("time")
    data = tag_data.get("datetime") if tag_data else None

    categoria = url.split("boatos.org/")[1].split("/")[0]

    return {
        "titulo": titulo,
        "boato": boato,
        "texto_completo": texto_completo,
        "data": data,
        "categoria": categoria,
        "url": url,
        "label": 1,
    }

# 1) coletar os links de todas as paginas
todos_links = []
for n in range(1, NUM_PAGINAS + 1):
    url = "https://www.boatos.org/" if n == 1 else f"https://www.boatos.org/page/{n}/"
    links = pegar_links(url)
    print(f"Pagina {n}: {len(links)} links")
    if not links:
        break
    todos_links.extend(links)
    time.sleep(2)

# 2) coletar o conteudo de cada noticia
noticias = []
for i, link in enumerate(todos_links):
    try:
        noticias.append(pegar_noticia(link))
    except Exception as e:
        print("Erro em", link, e)
    time.sleep(2)
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(todos_links)} coletadas")

df_boatos = pd.DataFrame(noticias)
df_boatos.to_csv("boatos_corpus.csv", index=False)
print(df_boatos.shape)
print(df_boatos[["titulo", "data", "categoria"]].head())

Pagina 1: 25 links
Pagina 2: 25 links
10/50 coletadas
20/50 coletadas
30/50 coletadas
40/50 coletadas
50/50 coletadas
(50, 7)
                                              titulo  \
0  Notícia sobre fábricas da Coca-Cola fechadas n...   
1  Vídeo em que Zico anuncia voto em Lula no segu...   
2  Vídeo em que Virginia Fonseca anuncia voto em ...   
3  Não há provas nem indícios de que Flávio Bolso...   
4  Vídeo de mulher usando suposta droga não é de ...   

                        data       categoria  
0  2026-10-13T21:17:28-03:00          brasil  
1  2026-10-13T20:22:39-03:00         esporte  
2  2026-10-13T19:18:33-03:00  entretenimento  
3  2026-10-13T18:26:17-03:00        politica  
4  2026-10-13T17:06:20-03:00        politica  


In [ ]:
for i in range(3):
    print("TÍTULO:", df_boatos["titulo"][i])
    print("BOATO:", df_boatos["boato"][i])
    print("-" * 60)

print(df_boatos["boato"].str.len().describe())

TÍTULO: Notícia sobre fábricas da Coca-Cola fechadas não tem relação com unidades da empresa no Brasil
BOATO: Boato – A Coca-Cola fechou quatro fábricas no Brasil e demitiu cerca de 1.400 funcionários por causa do governo Lula e do PT.
------------------------------------------------------------
TÍTULO: Vídeo em que Zico anuncia voto em Lula no segundo turno é fake criado por IA
BOATO: Boato – Zico anunciou que vai votar em Lula no segundo turno das eleições de 2026 e pediu que os internautas sigam um perfil nas redes sociais.
------------------------------------------------------------
TÍTULO: Vídeo em que Virginia Fonseca anuncia voto em Lula é falso e foi criado com IA
BOATO: Boato – Virginia Fonseca anunciou que vai votar em Lula no segundo turno das eleições de 2026 e afirmou que deixou de se posicionar por respeito à família de Zé Felipe.
------------------------------------------------------------
count     50.000000
mean     125.660000
std       25.642447
min       76.000000
25

In [ ]:
df_boatos["boato_limpo"] = df_boatos["boato"].str.replace(r"^Boato\s*[–-]\s*", "", regex=True)
print(df_boatos["boato_limpo"][0])

A Coca-Cola fechou quatro fábricas no Brasil e demitiu cerca de 1.400 funcionários por causa do governo Lula e do PT.
